# 02 | Coorte final e dataset supervisionado

Fluxo único e oficial: **auditar → gerar candidata → publicar → gerar artefato (partição 70/15/15 em blocos)**.

Contrato: `configs/final_cohort_v1.yaml`. Partição: `configs/supervised.yaml` (`temporal`).

Partição (anos 2019, 2023–2025): `train_ref` 0,70 · `val_ref` 0,15 · `test_ref` 0,15. Anos 2020–2022 → `test_pandemic`; 2026 → `exploracao`. Testes seguem fechados até o G9.

## Estratégia de memória

- Rode cada seção uma vez e desligue o interruptor correspondente na célula 2 (`False`) quando o produto já existir. Isso evita reprocessar tudo a cada abertura.
- A célula do artefato (célula 11) roda em **subprocesso** por padrão (`USAR_SUBPROCESSO = True`). O processo filho carrega a coorte inteira, escreve e **morre** — a RAM volta ao SO. O kernel fica limpo.
- Se preferir rodar no kernel, **reinicie o kernel** antes de executar a célula 11 (rode só a célula 2 e a 11 na mesma sessão).
- O monitor de RAM aparece em cada célula pesada. Se o RSS passar de ~70% da RAM física no modo kernel, pare e reinicie o kernel.

In [ ]:
import json, shutil, sys, os, gc, subprocess, hashlib
from pathlib import Path
import psutil

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "supervised.yaml").exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]

import pandas as pd
from IPython.display import display
from src.data.final_cohort_cli import main as cohort_cli
from src.data.supervised_artifact import gerar_artefato_supervisionado

def mem_gb():
    return psutil.Process().memory_info().rss / 1e9

def existe(p):
    return p.exists() and p.stat().st_size > 0

def hash_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

CAND = ROOT / "outputs/coorte_candidata"
PUB  = ROOT / "data/analytical/coorte_final.parquet"
ART  = ROOT / "data/analytical/dataset_supervisionado.parquet"
MAN  = ROOT / "data/analytical/manifesto_supervisionado.json"

print(f"RAM inicial: {mem_gb():.1f} GB")
print(f"candidata existe?         {existe(CAND / 'coorte_final.parquet')}")
print(f"coorte publicada existe?  {existe(PUB)}")
print(f"artefato existe?          {existe(ART)}")
print(f"manifesto existe?         {existe(MAN)}")

# >>> AQUI está a economia de memória <<<
# Ligue só o que ainda não existe. Tudo já gerado = False.
GERAR_CANDIDATA  = True   # vira False depois da primeira execução bem-sucedida
PUBLICAR         = True   # vira False depois da primeira execução bem-sucedida
GERAR_ARTEFATO   = True   # vira False depois da primeira execução bem-sucedida
USAR_SUBPROCESSO = True   # recomendado: libera a RAM ao terminar o artefato

REF_24_10 = {"train_ref": 556_462, "val_ref": 119_243,
             "test_ref": 119_245, "test_pandemic": 2_915_829}

# Sanidade: se o artefato existe e NÃO vamos regerar, confere o hash do manifesto.
if not GERAR_ARTEFATO and existe(ART) and existe(MAN):
    _man = json.loads(MAN.read_text(encoding="utf-8"))
    _esp = _man.get("dataset_sha256", "")
    _atu = hash_file(ART)
    _ok  = (_atu == _esp)
    print(f"artefato: cohort_kind = {_man.get('cohort_kind')}")
    print(f"artefato: features = {len(_man.get('feature_columns', []))}")
    print(f"artefato: hash {'OK' if _ok else 'DIVERGE'} (esperado {_esp[:12]}, obtido {_atu[:12]})")
    del _man, _esp, _atu, _ok

print("Raiz:", ROOT)

## 1. Auditoria (não remove nada)

In [ ]:
if GERAR_CANDIDATA:
    assert cohort_cli(["--mode", "audit", "--overwrite"]) == 0
else:
    print("Auditoria pulada (GERAR_CANDIDATA=False).")

_resumo = CAND / "resumo_regras.csv"
if _resumo.exists():
    display(pd.read_csv(_resumo))
else:
    print("resumo_regras.csv ainda não existe.")
del _resumo
gc.collect()
print(f"RAM após auditoria: {mem_gb():.1f} GB")

## 2. Candidata + validação
Gera `outputs/coorte_candidata/`. O waterfall precisa fechar (fonte = excluídas + candidata).

In [ ]:
if GERAR_CANDIDATA:
    assert cohort_cli(["--mode", "candidate", "--overwrite"]) == 0

_val_path   = CAND / "validacao_publicacao.json"
_water_path = CAND / "coorte_waterfall.csv"

if _val_path.exists() and _water_path.exists():
    val = json.loads(_val_path.read_text(encoding="utf-8"))
    rec = val["population_reconciliation"]
    print(rec)
    assert rec["closed"], "waterfall não fecha: não publique"
    display(pd.read_csv(_water_path))
else:
    val = rec = None
    print("validacao_publicacao.json ou coorte_waterfall.csv ainda não existe.")

del _val_path, _water_path
gc.collect()
print(f"RAM após candidata: {mem_gb():.1f} GB")

## 3. Publicar
Copia a candidata para `data/analytical/coorte_final.parquet` (com backup). O CLI não tem modo publish.

In [ ]:
if PUBLICAR:
    _src = CAND / "coorte_final.parquet"
    if not _src.exists():
        print(f"Nada para publicar: {_src} não existe (rode GERAR_CANDIDATA=True primeiro).")
    else:
        PUB.parent.mkdir(parents=True, exist_ok=True)
        if PUB.exists():
            shutil.copy2(PUB, PUB.with_suffix(".parquet.bak"))
        shutil.copy2(_src, PUB)

        _mp = PUB.with_name("coorte_final.manifest.json")
        shutil.copy2(CAND / "manifesto_coorte.json", _mp)

        _m = json.loads(_mp.read_text(encoding="utf-8"))
        _m["publication_performed"] = True
        _m["legacy_parity_reviewed"] = True
        _m["provisional_comparison_reviewed"] = True
        _mp.write_text(json.dumps(_m, indent=2, ensure_ascii=False), encoding="utf-8")
        del _mp, _m
    del _src

if PUB.exists():
    print(f"Coorte publicada: {PUB} ({PUB.stat().st_size/1e6:.1f} MB)")
else:
    print(f"Coorte ainda não publicada em {PUB}")

# Libera o que restou do kernel (dicts pequenos, mas evita lixo entre células).
for _n in ("val", "rec"):
    if _n in globals():
        del globals()[_n]
gc.collect()
try:
    import pyarrow as pa
    pa.default_memory_pool().release_unused()
except Exception:
    pass
print(f"RAM após publicar: {mem_gb():.1f} GB")

## 4. Dataset supervisionado + partição-mestre
Grava `dataset_supervisionado.parquet`, `partitions_master.parquet` e `manifesto_particao.json`. Determinístico (hash do `group_id`, seed 42). Leva alguns minutos.

**Se `USAR_SUBPROCESSO = True`** (recomendado), o artefato é gerado num processo filho que **morre ao terminar** — a RAM volta ao SO e o kernel não fica carregado. Você vê o progresso linha a linha; o resultado é lido do manifesto no fim.

In [ ]:
if GERAR_ARTEFATO:
    if USAR_SUBPROCESSO:
        _code = (
            "from pathlib import Path; "
            "from src.data.supervised_artifact import gerar_artefato_supervisionado; "
            "gerar_artefato_supervisionado(Path('.'), cohort_kind='final', "
            "exportar_particao=True, overwrite=True); "
            "print('OK')"
        )
        print("Rodando em subprocesso (memória liberada ao terminar o processo)...")
        _env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
        _proc = subprocess.Popen(
            [sys.executable, "-c", _code],
            cwd=str(ROOT),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace",
            bufsize=1, env=_env,
        )
        for _line in _proc.stdout:
            print(_line, end="", flush=True)
        _proc.stdout.close()
        _proc.wait()
        _rc = _proc.returncode
        del _proc, _code, _env
        if _rc != 0:
            raise RuntimeError(f"gerar_artefato_supervisionado falhou (código {_rc})")
    else:
        print("Rodando no kernel (vai consumir RAM; reinicie o kernel antes, se possível)...")
        gerar_artefato_supervisionado(ROOT, cohort_kind="final", exportar_particao=True, overwrite=True)

# Lê as contagens do manifesto — funciona nos dois caminhos e quando GERAR_ARTEFATO=False.
contagens = None
if MAN.exists():
    _man = json.loads(MAN.read_text(encoding="utf-8"))
    contagens = _man.get("partition_counts")
    del _man

if contagens is not None:
    tab = pd.DataFrame({"linhas": pd.Series(contagens), "ref_24_10": pd.Series(REF_24_10)})
    tab["igual_ref"] = tab["linhas"] == tab["ref_24_10"]
    display(tab)
else:
    print("Sem manifesto; contagens indisponíveis.")

gc.collect()
print(f"RAM após o artefato: {mem_gb():.1f} GB")

## 5. Conferência da razão 70/15/15
Percentuais só sobre os blocos dos anos compartilhados (os pandêmicos e 2026 ficam de fora por desenho).

In [ ]:
if contagens is not None:
    comp = {k: v for k, v in contagens.items() if k in ("train_ref", "val_ref", "test_ref")}
    pct = pd.Series(comp) / sum(comp.values())
    esperado = pd.Series({"train_ref": .70, "val_ref": .15, "test_ref": .15})
    display(pd.DataFrame({"pct": pct.round(4), "esperado": esperado}))
    assert ((pct - esperado).abs() < 0.005).all(), "razão fora de 70/15/15 (tolerância 0,5 p.p.)"
    print("OK: 70/15/15. Testes seguem fechados até o G9.")
else:
    print("Sem contagens; rode a célula do artefato primeiro.")